# 01 — Loading OHLCV Data

This notebook introduces the OHLCV format and how to load market data with `yfinance`.

OHLCV columns:
- Open
- High
- Low
- Close
- Volume

In [ ]:
import numpy as np
import pandas as pd

try:
    import yfinance as yf
    YF_AVAILABLE = True
except Exception:
    YF_AVAILABLE = False

print("yfinance available:", YF_AVAILABLE)

## 1) Download data from Yahoo Finance

If download fails (internet/rate limits), we automatically generate synthetic OHLCV.

In [ ]:
ticker = "AAPL"
start_date = "2021-01-01"
end_date = "2025-12-31"

df = None
if YF_AVAILABLE:
    try:
        df = yf.download(ticker, start=start_date, end=end_date, auto_adjust=False, progress=False)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = [c[0] for c in df.columns]
        df = df[["Open", "High", "Low", "Close", "Volume"]].dropna()
        if len(df) == 0:
            df = None
    except Exception:
        df = None

if df is None:
    np.random.seed(42)
    n = 900
    dates = pd.date_range(start="2022-01-01", periods=n, freq="B")
    base = 100 + np.cumsum(np.random.normal(0.02, 1.2, n))
    close = np.maximum(base, 1)
    open_ = close + np.random.normal(0, 0.6, n)
    high = np.maximum(open_, close) + np.abs(np.random.normal(0.4, 0.3, n))
    low = np.minimum(open_, close) - np.abs(np.random.normal(0.4, 0.3, n))
    volume = np.random.randint(1_000_000, 6_000_000, n)
    df = pd.DataFrame({
        "Open": open_,
        "High": high,
        "Low": low,
        "Close": close,
        "Volume": volume
    }, index=dates)
    print("Using synthetic OHLCV data fallback")
else:
    print(f"Downloaded {ticker} from Yahoo Finance")

df.head()

## 2) Basic structure checks

In [ ]:
print("Shape:", df.shape)
print("\nDtypes:")
print(df.dtypes)
print("\nDate range:", df.index.min(), "to", df.index.max())

assert all(col in df.columns for col in ["Open", "High", "Low", "Close", "Volume"])
assert df.isna().sum().sum() == 0
print("Data checks passed ✅")

## 3) Fast exploratory analysis

In [ ]:
summary = df.describe().T[["mean", "std", "min", "max"]]
summary

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
df["Close"].plot(ax=ax[0], title="Close Price")
df["Volume"].plot(ax=ax[1], title="Volume", color="tab:orange")
plt.tight_layout()

## 4) Save a cleaned copy

This creates a dataset you can reuse in later notebooks.

In [ ]:
from pathlib import Path

output_dir = Path("../../data")
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "ohlcv_sample.csv"
df.to_csv(output_path)
print("Saved:", output_path.resolve())

## 5) Exercises

1. Change ticker to `MSFT` or `NVDA` and compare volatility.
2. Add a new column `Return` using daily percentage change of `Close`.
3. Compute average volume by month (`resample('M')`).

Next notebook: `02_technical_indicators.ipynb`.